In [2]:
import geopandas as gpd
import os

operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'

#Read the files
index_walkability = gpd.read_parquet(f'{output_step3_path}/step3_index.parquet')
index_walkability = index_walkability.to_crs(operation_crs)

zones_girec = gpd.read_file(f'{input_file_path}/network_agreg/GEO_GIREC-SHP/GEO_GIREC.shp')
zones_girec = zones_girec.to_crs(operation_crs)

agglo_carreau = gpd.read_file(f'{input_file_path}/network_agreg/AGGLO_CARREAU_200-SHP/AGGLO_CARREAU_200.shp')
agglo_carreau = agglo_carreau.to_crs(operation_crs)

**GIREC**

In [5]:
# Spatial join
segments_girec = gpd.sjoin(index_walkability, zones_girec, how="inner", predicate="within")

# Columns to aggregate
cols = index_walkability.columns.to_list()

def weighted_mean(df, cols, weight_col):
    return (df[cols].multiply(df[weight_col], axis=0).sum() / df[weight_col].sum())

cols_to_agg = cols[3:]  # tes colonnes d'indicateurs

# Calcul pondéré
girec_stats = (
    segments_girec
        .groupby("OBJECTID")
        .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
        .reset_index()
)

# Merge back with zones_mmt polygons
zones_girec = zones_girec.merge(girec_stats, on="OBJECTID", how="left")

#Drop nan values 
zones_girec = zones_girec.dropna(subset=["indice_marchabilite"])

/var/folders/gh/bcpf72g151g22zg7mqv0s2qw0000gn/T/ipykernel_9140/374657167.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))


**Carreau 200**

In [7]:
segments_carreau

,segment_id,geometry,length,bruit,temperature,conflit_usage,canopee,lac_cours_deau,fontaines,espaces_ouverts,...,PARTEMP,H_P_REGG,D_POP_HA,D_EMP_HA,POP_TOT_GG,EMP_TOT_GG,POP_TOT_00,EMP_TOT_00,GEOM_AREA,GEOM_LEN
1,000001,"LINESTRING (2500439.51 1114635.486, 2500441.15...",6.863,0.0,1.0000,0.2500,0.523,0.0,0.0,0.0,...,0.000000,NaN,23.25,0.00,95,0,93,0,40000.0,800.0
2,000002,"LINESTRING (2501878.845 1118360.375, 2501878.5...",27.086,0.0,1.0000,0.2500,0.000,1.0,0.0,1.0,...,NaN,NaN,0.00,0.00,0,0,0,0,40000.0,800.0
3,000003,"LINESTRING (2501886.429 1118347.344, 2501883.9...",15.077,0.0,1.0000,0.2500,0.000,1.0,0.0,1.0,...,NaN,NaN,0.00,0.00,0,0,0,0,40000.0,800.0
4,000004,"LINESTRING (2496591.651 1117884.916, 2496591.6...",4.338,0.0,1.0000,0.1668,0.008,0.0,0.0,0.0,...,0.043478,NaN,297.00,13.50,1230,122,1188,54,40000.0,800.0
5,000005,"LINESTRING (2496593.499 1117892.836, 2496592.0...",4.600,0.0,1.0000,0.1668,0.026,0.0,0.0,0.0,...,0.043478,NaN,297.00,13.50,1230,122,1188,54,40000.0,800.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
287269,287269,"LINESTRING (2501306.982 1125777.835, 2501311.2...",23.452,0.0,0.2344,0.0000,1.000,0.0,0.0,0.0,...,0.096386,NaN,18.75,2.00,82,0,75,8,40000.0,800.0
287270,287270,"LINESTRING (2496322.642 1112381.173, 2496327.6...",11.457,1.0,0.1861,1.0000,1.000,1.0,0.0,0.0,...,0.135417,NaN,20.75,3.25,89,11,83,13,40000.0,800.0
287271,287271,"LINESTRING (2496332.748 1112386.57, 2496334.29...",1.891,1.0,0.2205,1.0000,1.000,1.0,0.0,0.0,...,0.135417,NaN,20.75,3.25,89,11,83,13,40000.0,800.0
287272,287272,"LINESTRING (2496309.103 1112372.054, 2496312.4...",8.110,1.0,0.1518,1.0000,1.000,1.0,0.0,0.0,...,0.135417,NaN,20.75,3.25,89,11,83,13,40000.0,800.0


In [8]:
# Spatial join: assign each segment to a carreau (grid cell)
segments_carreau = gpd.sjoin(index_walkability, agglo_carreau, how="inner", predicate="within")

# Aggregate by mean
carreau_stats = (
    segments_carreau
    .groupby("GRID_ID")
    .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))
    .reset_index()
)

# Merge back to grid polygons
agglo_carreau = agglo_carreau.merge(carreau_stats, on="GRID_ID", how="left")

# Drop rows with missing values (optional)
agglo_carreau = agglo_carreau.dropna(subset=["indice_marchabilite"])

/var/folders/gh/bcpf72g151g22zg7mqv0s2qw0000gn/T/ipykernel_9140/4071491662.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda d: weighted_mean(d, cols_to_agg, "length"))


**Export**

In [9]:
#save the file 
zones_girec.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_girec.gpkg"), driver="GPKG")
zones_girec.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_girec.parquet')

agglo_carreau.to_crs(target_crs).to_file(os.path.join(output_step3_path, "step3_aggregated_index_carreau200.gpkg"), driver="GPKG")
agglo_carreau.to_crs(target_crs).to_parquet(f'{output_step3_path}/step3_aggregated_index_carreau200.parquet')

In [11]:
zones_girec

,OBJECTID,NOM,NO_COM_FED,NO_COMM,CODE_SECT,SECT_VILLE,NUMERO,CD_SS_SECT,SHAPE_AREA,SHAPE_LEN,...,zone_apaisee,zone_pietonne,vitesse,walk_index,walk_index_unweighted,indice_marchabilite,Classe_Commodité,Classe_Attractivité,Classe_Infrastructure,Classe_Sécurité
0,31,Roulave,6620,20,00,None,2000020,020,1.579804e+06,7913.415859,...,0.000000,0.000000,0.702850,0.388557,0.388557,0.388557,0.518545,0.021357,0.528117,0.454982
1,32,Essertines,6620,20,00,None,2000040,040,7.162025e+05,4123.464182,...,0.000000,0.000000,0.589226,0.425055,0.425055,0.425055,0.632152,0.096223,0.443955,0.424801
2,33,La Tuilière,6620,20,00,None,2000010,010,1.333054e+06,6623.155220,...,0.000000,0.000000,0.657926,0.317296,0.317296,0.317296,0.394931,0.026821,0.436470,0.444260
3,1,Signal,6607,7,00,None,0700080,080,8.244912e+05,4110.832189,...,0.121723,0.000000,0.955566,0.367897,0.367897,0.367897,0.456784,0.012986,0.446339,0.556069
4,2,Veyrier - Marais,6645,48,00,None,4500060,060,6.213745e+05,4446.603843,...,0.026763,0.000000,0.916574,0.402385,0.402385,0.402385,0.485856,0.057968,0.492350,0.515776
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
470,439,Saint-Paul,6617,17,00,None,1700011,011,1.358225e+05,1669.972115,...,0.000000,0.000000,0.098507,0.449104,0.449104,0.449104,0.560185,0.250974,0.547298,0.219142
471,440,Châtillon,6607,7,00,None,0700020,020,1.296117e+06,5130.673474,...,0.000000,0.000000,0.783134,0.451321,0.451321,0.451321,0.610342,0.009075,0.625187,0.476878
472,441,Narly - Lécherette,6618,18,00,None,1800041,041,6.575811e+05,4855.945566,...,0.646689,0.000000,0.936878,0.515625,0.515625,0.515625,0.476752,0.161331,0.543514,0.676300
473,442,Les Délices,6621,21,03,DELICES GROTTES,2103010,010,1.508622e+05,1613.385489,...,0.776825,0.000014,0.776839,0.528629,0.528629,0.528629,0.519275,0.337601,0.418797,0.526085
